<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-05-bigquery-ml/lesson-5.4-registry/notebooks/GCP_Capstone_5.4_Registry.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.4 BigQuery → Vertex AI — Model Registry, DataFrames & Feature Engineering
**Netsetos GenAI Engineering — GCP Capstone**

Export models to production, explore with pandas-at-scale, engineer features that keep models accurate.


## Setup


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Ensure this module's datasets exist (idempotent -- BigQuery never auto-creates them).
# NOTE: cells that read rag_data.document_features need Lesson 5.1 run first.
for _ds in ('rag_data', 'ml_models'):
    _d = bigquery.Dataset(f'{PROJECT_ID}.{_ds}'); _d.location = 'US'
    client.create_dataset(_d, exists_ok=True)

# Preflight (idempotent): Vertex AI API for model_registry='vertex_ai', and a US bucket
# for EXPORT MODEL (Cell 3). Both are quick no-ops if already present.
import subprocess
subprocess.run(['gcloud', 'services', 'enable', 'aiplatform.googleapis.com',
                '--project', PROJECT_ID], check=False)
_models_bucket = f'{PROJECT_ID}-models'
if subprocess.run(['gcloud', 'storage', 'buckets', 'describe', f'gs://{_models_bucket}',
                   '--project', PROJECT_ID], capture_output=True).returncode:
    subprocess.run(['gcloud', 'storage', 'buckets', 'create', f'gs://{_models_bucket}',
                    '--project', PROJECT_ID, '--location', 'US',
                    '--uniform-bucket-level-access'], check=False)

def run_query(sql):
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "OK"}')

def train_model(sql, minutes=25):
    """CREATE MODEL for a managed-backend model (BOOSTED_TREE / DNN / AutoML). These
    provision compute and take several minutes, so poll with progress instead of a silent
    blocking wait, and fail loud if the job stalls past `minutes`."""
    import time
    job = client.query(sql)
    print(f'Training on the managed backend (job {job.job_id}) - takes several minutes; polling...')
    t0 = time.monotonic()
    while not job.done():
        if time.monotonic() - t0 > minutes * 60:
            raise TimeoutError(f'Model still {job.state} after {minutes} min - '
                               f'check BigQuery > Job history (job {job.job_id}).')
        time.sleep(30)
        print(f'  ...{int(time.monotonic() - t0)}s elapsed, state={job.state}')
    job.result()  # surfaces any training error
    print(f'Model trained in ~{int(time.monotonic() - t0)}s')

print(f'Connected to {PROJECT_ID}')


## Cell 1: Register Model in Vertex AI


In [ ]:
# Prerequisite: run Lesson 5.1 first to create rag_data.document_features (BigQuery tables persist per project).
# Train + register in one step
train_model(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['document_type'],
  model_registry = 'vertex_ai',
  vertex_ai_model_id = 'documind_doc_classifier',
  vertex_ai_model_version_aliases = ['v2', 'latest'],
  max_iterations = 50,
  enable_global_explain = TRUE
) AS
SELECT page_count, chunk_count, total_word_count,
       file_size_mb, content_type, avg_chunk_size, document_type
FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Model trained and registered in Vertex AI')


## Cell 2: Evaluate + Global Explain


In [ ]:
# Evaluate
print('=== Classification Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`)'))

# Feature importance
print('\n=== Feature Importance (Shapley) ===')
print(run_query(f'SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`)'))

# Feature stats
print('\n=== Feature Info ===')
print(run_query(f'SELECT * FROM ML.FEATURE_INFO(MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`)'))


## Cell 3: Export Model to GCS


In [ ]:
# Export model artifacts (BOOSTED_TREE exports as XGBoost Booster files).
# Same-location rule: the BigQuery dataset and the GCS bucket must be in the SAME
# location. These datasets are US multi-region, so the setup cell created the bucket
# (ARIMA_PLUS is not exportable at all - see the deployability table on the lesson page.)
try:
    run_ddl(f'''
    EXPORT MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`
    OPTIONS(URI = 'gs://{PROJECT_ID}-models/doc_classifier/v2/')
    ''')
    print('Model exported to GCS')
except Exception as e:
    print(f'Export needs the dataset and the bucket in the same location: {e}')


## Cell 4: BigQuery DataFrames — Explore Data


In [ ]:
!pip install -q bigframes

import bigframes.pandas as bpd

bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = 'US'

# Read DocuMind data
docs = bpd.read_gbq(f'{PROJECT_ID}.rag_data.document_features')

# Pandas operations at BigQuery scale
stats = (
    docs.groupby('document_type')
    .agg({
        'page_count': 'mean',
        'processing_cost_usd': 'sum',
        'doc_id': 'count'
    })
    .rename(columns={'doc_id': 'doc_count'})
    .sort_values('doc_count', ascending=False)
)

print('=== Document Stats ===')
print(stats.peek(10))

# Show the generated SQL
print('\n=== Generated SQL ===')
print(stats.sql)


## Cell 5: bigframes.ml — Scikit-Learn Pipeline

> `pipeline.fit` trains an XGBoost model on the managed backend, so this cell also takes ~5-15 minutes (like the boosted-tree cells) - it is not stuck.

In [ ]:
from bigframes.ml.pipeline import Pipeline
from bigframes.ml.compose import ColumnTransformer
from bigframes.ml.preprocessing import StandardScaler, OneHotEncoder
from bigframes.ml.ensemble import XGBClassifier
from bigframes.ml.model_selection import train_test_split

# Prepare data
X = docs.drop(columns=['document_type', 'doc_id', 'title'])
y = docs[['document_type']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Build pipeline
preprocessor = ColumnTransformer([
    ('scale', StandardScaler(),
     ['page_count', 'total_word_count', 'file_size_mb']),
    ('encode', OneHotEncoder(),
     ['content_type'])
])

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', XGBClassifier())
])

# Train (generates BQML CREATE MODEL)
pipeline.fit(X_train, y_train)
score = pipeline.score(X_test, y_test)
print('=== bigframes ML Score ===')
print(score.to_pandas())

# Save as BQML model
pipeline.to_gbq(f'{PROJECT_ID}.ml_models.doc_classifier_bf', replace=True)
print('Pipeline saved as BQML model')


## Cell 6: TRANSFORM with Full Feature Engineering

> Note: this model's TRANSFORM uses `ML.FEATURE_CROSS`/`SAFE_DIVIDE`, which aren't in the export-supported function list, so it is train / evaluate / registry-only (not exportable for online serving). Cell 3 exports the plain `doc_classifier_v2` model instead.

In [ ]:
# Model with comprehensive TRANSFORM clause
train_model(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier_transform`
TRANSFORM (
  ML.STANDARD_SCALER(total_word_count) OVER() AS words_scaled,
  ML.MIN_MAX_SCALER(page_count) OVER() AS pages_scaled,
  ML.QUANTILE_BUCKETIZE(file_size_mb, 5) OVER() AS size_bucket,
  ML.FEATURE_CROSS(STRUCT(content_type AS ctype, CAST(page_count > 10 AS STRING) AS long_doc)) AS type_length,
  LOG(total_word_count + 1) AS log_words,
  SAFE_DIVIDE(chunk_count, page_count) AS chunks_per_page,
  document_type
)
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['document_type'],
  enable_global_explain = TRUE,
  model_registry = 'vertex_ai',
  vertex_ai_model_id = 'documind_classifier_transform'
) AS
SELECT * FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Model with TRANSFORM trained and registered')

# Compare feature importance
print('\n=== Transform Model Feature Importance ===')
print(run_query(f'SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `{PROJECT_ID}.ml_models.doc_classifier_transform`)'))


## Cell 7: Window Features - Time-Based Feature Table

> This cell builds the feature table **once, by hand**. The scheduled query that rebuilds it every morning and the materialized-view alternative are both on the lesson page (Step 9) - this notebook creates neither.


In [ ]:
# Create time-based feature table
try:
    run_ddl(f'''
    CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.doc_time_features` AS
    WITH daily AS (
      SELECT
        doc_id,
        DATE_ADD(DATE '2026-01-01', INTERVAL SAFE_CAST(REGEXP_EXTRACT(doc_id, r'[0-9]+') AS INT64) DAY) AS query_date,
        page_count AS daily_metric
      FROM `{PROJECT_ID}.rag_data.document_features`
    )
    SELECT
      doc_id, query_date, daily_metric,
      AVG(daily_metric) OVER (
        PARTITION BY doc_id ORDER BY query_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
      ) AS rolling_avg_3d,
      LAG(daily_metric, 1) OVER (
        PARTITION BY doc_id ORDER BY query_date
      ) AS prev_day,
      RANK() OVER (
        PARTITION BY query_date ORDER BY daily_metric DESC
      ) AS daily_rank
    FROM daily
    ''')
    print('Time-based features created')
    print(run_query(f'SELECT * FROM `{PROJECT_ID}.rag_data.doc_time_features` LIMIT 5'))
except Exception as e:
    print(f'Note: {e}')


## Lesson 5.4 Complete

**What this notebook actually ran:**
- `model_registry='vertex_ai'` for auto-registration (Cells 1 and 6)
- `ML.EVALUATE`, `ML.GLOBAL_EXPLAIN`, `ML.FEATURE_INFO` for metrics and feature importance (Cell 2)
- `EXPORT MODEL` to a GCS bucket in the dataset's location (Cell 3)
- BigQuery DataFrames for pandas-at-scale exploration (Cell 4)
- `bigframes.ml` Pipeline - scikit-learn syntax, BQML under the hood (Cell 5)
- `TRANSFORM` with 6 preprocessing functions (Cell 6)
- Window functions (`AVG OVER`, `LAG`, `RANK`) written into a feature **table** (Cell 7)

**Covered in the lesson but NOT run here - do not tick these off yet:**
- **Deploy to a Vertex AI endpoint (Python).** The code is on the lesson page, Step 4; you run it in the **Practice Lab notebook, Exercise 7**. It stays out of this notebook because a deployed endpoint bills a warm machine 24x7 (~$55/month, about Rs 4,675 at USD_INR = 85) whether you call it or not.
- **Scheduled query and materialized view.** Both are on the lesson page, **Step 9** - the `bq ... query --schedule` command that rebuilds the feature table every morning at 06:00, and the `CREATE MATERIALIZED VIEW` alternative for plain aggregates. Cell 7 above builds that table once, manually.

**Module 5 today - four lessons, with a fifth on the way:**
- 5.1: CREATE MODEL (regression, classification, clustering)
- 5.2: ARIMA_PLUS + AI.FORECAST (forecast, anomaly, decompose)
- 5.3: AI functions (AI.GENERATE, VECTOR_SEARCH, RAG-in-SQL)
- 5.4: Production (registry, DataFrames, feature engineering) - you are here
- 5.5 (planned): **Engineer Retrieval Features** - chunk metadata, AI.GENERATE_TABLE, BQML routing features, Knowledge Catalog quality gates

**Next: Lesson 5.5.** It points the feature engineering you just did at retrieval - the window features and TRANSFORM habits become chunk metadata that ships as Vector Search **restricts**, plus a BQML **routing classifier** that decides which index a query should hit. Module 6 (Function Calling & Tool Use) comes after the module closes.
